In [ ]:
MODELS = [
    "sentence-transformers@sentence-t5-xxl", # No prompt.
    "google@embeddinggemma-300m", # Task-tuned.
    "tencent@KaLM-Embedding-Gemma3-12B-2511" # Instruct-based.
]

NAMES = {
    "sentence-transformers@sentence-t5-xxl": "ST5 XXL",
    "google@embeddinggemma-300m": "EmbeddingGemma",
    "tencent@KaLM-Embedding-Gemma3-12B-2511": "KaLM v2"
}

In [ ]:
import pandas as pd

# Datasets

## GoEmotions

In [ ]:
import io

import requests

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/google-research/google-research/refs/heads/master/goemotions/data"

In [ ]:
emotions = requests.get(f"{DATA_URL}/emotions.txt").text.splitlines()
ekman_mapping = requests.get(f"{DATA_URL}/ekman_mapping.json").json()

inverse_ekman_mapping = {e: k for k, v in ekman_mapping.items() for e in v}

# Visualization

## Setup

In [ ]:
import pickle


with open("viz.pkl", "rb") as f:
    obj = pickle.load(f)

    data_dict, manifold_dict = obj["data_dict"], obj["manifold_dict"]

In [ ]:
for key in ["NRC-VAD", "NRC-EIL", "GoEmotions"]:
    data_dict[key] = pd.DataFrame.from_dict(data_dict[key])

for name in MODELS:
    for key, column in zip(["NRC-VAD", "NRC-EIL", "GoEmotions"], ["term", "word", "sentence_text"]):
        manifold_dict[name][key] = pd.DataFrame.from_dict(manifold_dict[name][key])

## Plotting

In [ ]:
import numpy as np

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm

import seaborn as sns

In [ ]:
FIGSIZE = (8, 8 / ((1 + 5 ** 0.5) / 2))

In [ ]:
def convert_nrc_vad(visual_df, manifold_df, sz=500, pct=0.01, seed=42):
    rng = np.random.default_rng(seed)

    idx = rng.choice(np.arange(len(visual_df)), size=min(sz, len(visual_df)) if sz is not None else np.floor(pct * len(visual_df)).astype(int), replace=False)

    X_df, y_df = manifold_df.copy().iloc[idx], visual_df.copy().iloc[idx]

    X = X_df.to_numpy()

    colors_rgb = y_df[["valence", "arousal", "dominance"]].to_numpy()
    colors_rgb = (colors_rgb + 1.0) / 2.0
    colors_hsv = mpl.colors.rgb_to_hsv(colors_rgb)
    y = colors_hsv.copy()

    return X, y

def plot_nrc_vad(canvas, X, y, style=None):
    canvas.scatter(X[:, 0], X[:, 1], c=y[:, 0], alpha=0.5, cmap="twilight_shifted", marker=style)

    canvas.set_axisbelow(True)
    canvas.grid(linestyle="--")

def decorate_nrc_vad(canvas):
    cmap, norm = mpl.colormaps["twilight_shifted"], mcolors.Normalize(vmin=0.0, vmax=1.0)
    cbar = fig.colorbar(cm.ScalarMappable(cmap=cmap, norm=norm), cax=canvas)
    cbar.set_ticks([0.0, 0.33, 0.66, 1.0])
    cbar.set_ticklabels(['R', 'G', 'B', 'R'])

In [ ]:
def convert_nrc_eil(visual_df, manifold_df, imbalanced=True, sz=500, pct=0.1, seed=42):
    rng = np.random.default_rng(seed)

    X_df, y_df = manifold_df.copy(), visual_df.copy()

    pos_emos, neg_emos = ["joy", "trust", "anger", "anticipation"], ["sadness", "disgust", "fear", "surprise"]

    pos_idx, neg_idx = ((y_df[pos_emos] > 0.0).sum(axis=1) == 1) & ((y_df[neg_emos] == 0.0).all(axis=1)), ((y_df[pos_emos] == 0.0).all(axis=1)) & ((y_df[neg_emos] > 0.0).sum(axis=1) == 1)

    num_pts = min(sz, len(visual_df)) if sz is not None else min(sum(idx), np.floor(pct * len(visual_df)).astype(int))

    if imbalanced:
        idx = pos_idx | neg_idx

        idx_viz = rng.choice(np.nonzero(idx)[0], size=num_pts, replace=False)
    else:
        idx_viz = []

        for idx in [pos_idx, neg_idx]:
            sub_idx = rng.choice(np.nonzero(idx)[0], size=min(sum(idx), num_pts // 2), replace=False)

            idx_viz.extend(sub_idx)

    X_df, y_df = X_df.iloc[idx_viz], y_df.iloc[idx_viz]

    pos_idx, neg_idx = pos_idx.iloc[idx_viz], neg_idx.iloc[idx_viz]

    X = X_df.to_numpy()

    y = pos_idx.astype(int) * (y_df[pos_emos].sum(axis=1) / 2) + neg_idx.astype(int) * (0.5 + y_df[neg_emos].sum(axis=1) / 2)

    return X, y

def plot_nrc_eil(canvas, X, y, style=None):
    canvas.scatter(X[:, 0], X[:, 1], c=y[:], alpha=0.5, cmap="coolwarm", marker=style)

    canvas.set_axisbelow(True)
    canvas.grid(linestyle="--")

def decorate_nrc_eil(canvas):
    cmap, norm = mpl.colormaps["coolwarm"], mcolors.Normalize(vmin=-1.0, vmax=1.0)
    cbar = fig.colorbar(cm.ScalarMappable(cmap=cmap, norm=norm), cax=canvas)
    cbar.set_ticks([-1.0, 1.0])
    cbar.set_ticklabels(["Negative", "Positive"])

In [ ]:
def convert_goemotions(visual_df, manifold_df, imbalanced=True, sz=500, pct=0.015, seed=42):
    rng = np.random.default_rng(seed)

    X_df, y_df = manifold_df.copy(), visual_df.copy()

    num_pts = min(sz, len(visual_df)) if sz is not None else np.floor(pct * len(visual_df)).astype(int)

    if imbalanced:
        idx = rng.choice(np.arange(len(visual_df)), size=num_pts, replace=False)
    else:
        num_cls = len(ekman_mapping.keys())

        idx = []

        for emo in ekman_mapping.keys():
            emo_idx = visual_df["macro_category"] == emo

            idx_viz = rng.choice(np.nonzero(emo_idx)[0], size=min(sum(emo_idx), num_pts // num_cls), replace=False)

            idx.extend(idx_viz)

    X_df, y_df = X_df.iloc[idx], y_df.iloc[idx]

    X = X_df.to_numpy()

    y = y_df["macro_category"].to_list()
    y = [list(ekman_mapping.keys()).index(e) for e in y]

    return X, y

def plot_goemotions(canvas, X, y, style=None):
    canvas.scatter(X[:, 0], X[:, 1], color=list(map(lambda i: sns.color_palette("deep")[i], y)), alpha=0.5, marker=style)

    canvas.set_axisbelow(True)
    canvas.grid(linestyle="--")

def decorate_goemotions(canvas):
    num_colors = len(ekman_mapping.keys())
    colors = sns.color_palette("deep", n_colors=num_colors)
    cmap = mcolors.ListedColormap(colors)
    norm = mcolors.Normalize(vmin=0.0, vmax=float(num_colors))
    cbar = fig.colorbar(cm.ScalarMappable(cmap=cmap, norm=norm), cax=canvas)
    cbar.set_ticks([i + 0.5 for i in range(num_colors)])
    cbar.set_ticklabels(list(map(lambda x: x.title(), ekman_mapping.keys())))

In [ ]:
from datetime import datetime


def update_lims(ax, global_xlim, global_ylim, reset=False):
    (xbottom, xtop), (ybottom, ytop) = ax.get_xlim(), ax.get_ylim()
    if reset:
        return (xbottom, xtop), (ybottom, ytop)
    return (min(xbottom, global_xlim[0]), max(xtop, global_xlim[1])), (min(ybottom, global_ylim[0]), max(ytop, global_ylim[1]))

seed = 42

fig, axes = plt.subplots(
    nrows=3,
    ncols=4,
    width_ratios=[1, 1, 1, 0.05],
    figsize=(3 * FIGSIZE[0], 3.05 * FIGSIZE[1])
)

xlims, ylims = [None] * 3, [None] * 3

for j, (name, style) in enumerate(zip(MODELS, ['^', 's', 'o'])):
    X, y = convert_nrc_vad(data_dict["NRC-VAD"], manifold_dict[name]["NRC-VAD"], seed=seed)
    plot_nrc_vad(axes[0, j], X, y, style=None)
    decorate_nrc_vad(axes[0, 3])

    X, y = convert_nrc_eil(data_dict["NRC-EIL"], manifold_dict[name]["NRC-EIL"], imbalanced=False, seed=seed)
    plot_nrc_eil(axes[1, j], X, y, style=None)
    decorate_nrc_eil(axes[1, 3])

    X, y = convert_goemotions(data_dict["GoEmotions"], manifold_dict[name]["GoEmotions"], imbalanced=False, seed=seed)
    plot_goemotions(axes[2, j], X, y, style=None)
    decorate_goemotions(axes[2, 3])

    for i, ax in enumerate(axes[:, j]):
        global_xlim, global_ylim = update_lims(ax, xlims[i], ylims[i], reset=j == 0)
        xlims[i], ylims[i] = global_xlim, global_ylim

for i in range(3):
    for j in range(3):
        axes[i, j].set_xlim(xlims[i])
        axes[i, j].set_ylim(ylims[i])

for ax in axes.flatten():
    ax.tick_params(labelsize=14)

for ax, label in zip(axes[:, 0], ["NRC-VAD", "NRC-EIL", "GoEmotions"]):
    ax.set_ylabel(label, weight="bold", fontsize=18, rotation=90)

for ax, label in zip(axes[0, :], list(map(lambda x: NAMES[x], MODELS))):
    ax.set_title(label, family="monospace", weight="bold", fontsize=18)

fig.savefig(f"matplotlib_{datetime.now().isoformat(timespec="seconds")}.pdf", transparent=True, bbox_inches="tight")

fig.show()